# QAOA Ising Depth Sweep -- Colab Driver

Thin driver notebook: installs pinned dependencies, clones this repo (on
Colab), and calls straight into `src/experiment.py`'s `run_depth_sweep`. No
sweep or plotting logic is duplicated here -- see `src/experiment.py` for
the actual implementation and `notebooks/analysis.ipynb` for the plots,
which loads the results this notebook saves under `results/` (no Colab
dependency, doesn't recompute anything).

This notebook also runs when opened locally (outside Colab) against an
already-checked-out repo with `requirements.txt` installed -- the setup
cell below detects the environment and skips the pip-install/clone step
in that case.

## Setup

**On Colab**: if you want the GPU device path (`device="GPU"` below,
`qiskit-aer-gpu` instead of `qiskit-aer`), select
**Runtime -> Change runtime type -> GPU** *before* running the cell below.

In [ ]:
import os
import sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/raumsie/qaoa-ising.git"

if IN_COLAB:
    # Pinned versions -- must match requirements.txt at the repo root; keep
    # these two in sync if requirements.txt changes.
    !pip install -q qiskit==2.5.0 qiskit-aer==0.17.2 qiskit-algorithms==0.4.0 numpy==2.5.1 scipy==1.18.0 matplotlib==3.11.1

    # --- GPU alternative (commented out) ---------------------------------
    # Requires a GPU runtime (see markdown cell above) and swaps plain
    # qiskit-aer for the CUDA-enabled build. Uninstall qiskit-aer first to
    # avoid a conflicting parallel install:
    #
    # !pip uninstall -y qiskit-aer
    # !pip install -q qiskit-aer-gpu==0.17.2
    # -----------------------------------------------------------------------

    if not os.path.isdir("qaoa-ising"):
        !git clone {REPO_URL}
    %cd qaoa-ising
    repo_root = os.getcwd()
else:
    # Local run: repo is already checked out and requirements.txt already
    # installed in the current environment --
    # just make sure `src` can be found whether running
    # from the repo root or from notebooks/.
    def _find_repo_root(start):
        d = os.path.abspath(start)
        for _ in range(5):
            if os.path.isdir(os.path.join(d, "src")) and os.path.isfile(os.path.join(d, "requirements.txt")):
                return d
            d = os.path.dirname(d)
        raise RuntimeError(f"Could not locate qaoa-ising repo root from {start}")

    repo_root = _find_repo_root(os.getcwd())
    print(
        "Not running in Colab -- assuming requirements.txt is already "
        "installed in the current environment; skipping pip install/git clone."
    )

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f"IN_COLAB={IN_COLAB}, repo_root=./{os.path.basename(repo_root)}")

In [2]:
from src.experiment import run_depth_sweep, records_to_dicts
from src.ising_model import generate_test_instances

## Run the depth sweep

Sweeps every (instance) x (QAOA depth `p`) x (optimizer) combination
via `run_depth_sweep` and records `epsilon(p) = (E_qaoa - E_0)/|E_0|`
against the exact-diagonalization baseline for each point.

The constants below are deliberately small for a **fast first run**
(a few minutes on CPU). Increase them at your leisure.

In [ ]:
import json
import time

# MINIMIZE_OPTIONS caps optimizer cost for a fast first run (maxiter for
# COBYLA, maxfun for L-BFGS-B). Set to None for an uncapped run once you
# have the time/GPU budget -- can take well over an hour on CPU otherwise.
P_VALUES = range(1, 4)
N_RESTARTS = 2                                 # full sweep: 5
MINIMIZE_OPTIONS = {"maxiter": 50, "maxfun": 60}  # full sweep: None
OPTIMIZER_METHODS = ("COBYLA", "L-BFGS-B")
DEVICE = "CPU"                                 # "GPU" needs qiskit-aer-gpu + Colab GPU runtime
SEED = 104                                      # reproducible end-to-end (see experiment.py)
N_SPINS = 6                                    # (qubits) generate_test_instances default

instances = generate_test_instances(n_spins=N_SPINS)
print("Instances:", list(instances.keys()))

t0 = time.time()
records = run_depth_sweep(
    instances=instances,
    p_values=P_VALUES,
    optimizer_methods=OPTIMIZER_METHODS,
    device=DEVICE,
    n_restarts=N_RESTARTS,
    minimize_options=MINIMIZE_OPTIONS,
    seed=SEED,
    verbose=True,
)
elapsed = time.time() - t0
print(f"\nSweep complete: {len(records)} records in {elapsed:.1f}s (device={DEVICE})")

## Save results

Saved as JSON (via `records_to_dicts`) under `results/` so
`notebooks/analysis.ipynb` can load and plot them without recomputing
anything (and without any Colab dependency).

In [ ]:
RESULTS_DIR = os.path.join(repo_root, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

output = {
    "config": {
        "p_values": list(P_VALUES),
        "n_restarts": N_RESTARTS,
        "optimizer_methods": list(OPTIMIZER_METHODS),
        "minimize_options": MINIMIZE_OPTIONS,
        "device": DEVICE,
        "seed": SEED,
        "n_spins": N_SPINS,
        "instance_names": list(instances.keys()),
        "wall_time_s_total": elapsed,
    },
    "records": records_to_dicts(records),
}

results_path = os.path.join(RESULTS_DIR, "depth_sweep_results.json")
with open(results_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"Saved {len(records)} records to {os.path.relpath(results_path, repo_root)}")

## Optional: CPU-vs-GPU wall-clock comparison

Runs a small matched subset of the sweep on `device="GPU"` and `device="CPU"`
and saves both timings so `analysis.ipynb` can show a CPU-vs-GPU bar chart.
This only works on a Colab GPU runtime with `qiskit-aer-gpu` installed (see
the setup cell above) -- it skips gracefully (prints a message, doesn't
raise) anywhere else, e.g. a local CPU-only run.

**Must run the main sweep before this cell**

In [ ]:
GPU_TIMING_P_VALUES = range(1, 4)
GPU_TIMING_N_RESTARTS = 2
GPU_TIMING_MINIMIZE_OPTIONS = {"maxiter": 50, "maxfun": 60}
GPU_TIMING_N_SPINS = N_SPINS                   # edit to size this test independently of the main sweep
GPU_TIMING_INSTANCES = {"uniform_FM": generate_test_instances(n_spins=GPU_TIMING_N_SPINS)["uniform_FM"]}

gpu_results_path = os.path.join(RESULTS_DIR, "gpu_timing_results.json")

try:
    t0 = time.time()
    gpu_records = run_depth_sweep(
        instances=GPU_TIMING_INSTANCES,
        p_values=GPU_TIMING_P_VALUES,
        optimizer_methods=OPTIMIZER_METHODS,
        device="GPU",
        n_restarts=GPU_TIMING_N_RESTARTS,
        minimize_options=GPU_TIMING_MINIMIZE_OPTIONS,
        seed=SEED,
        verbose=True,
    )
    gpu_elapsed = time.time() - t0

    # Matched CPU-device subset (same instance/p/optimizer/restarts/seed)
    t0 = time.time()
    cpu_records = run_depth_sweep(
        instances=GPU_TIMING_INSTANCES,
        p_values=GPU_TIMING_P_VALUES,
        optimizer_methods=OPTIMIZER_METHODS,
        device="CPU",
        n_restarts=GPU_TIMING_N_RESTARTS,
        minimize_options=GPU_TIMING_MINIMIZE_OPTIONS,
        seed=SEED,
        verbose=True,
    )
    cpu_elapsed = time.time() - t0

    gpu_output = {
        "config": {
            "p_values": list(GPU_TIMING_P_VALUES),
            "n_restarts": GPU_TIMING_N_RESTARTS,
            "minimize_options": GPU_TIMING_MINIMIZE_OPTIONS,
            "optimizer_methods": list(OPTIMIZER_METHODS),
            "instance_names": list(GPU_TIMING_INSTANCES.keys()),
            "n_spins": GPU_TIMING_N_SPINS,
            "seed": SEED,
        },
        "gpu_records": records_to_dicts(gpu_records),
        "cpu_records": records_to_dicts(cpu_records),
        "gpu_wall_time_s_total": gpu_elapsed,
        "cpu_wall_time_s_total": cpu_elapsed,
    }
    with open(gpu_results_path, "w") as f:
        json.dump(gpu_output, f, indent=2)

    print(f"Saved CPU-vs-GPU timing comparison to {os.path.relpath(gpu_results_path, repo_root)}")
    print(f"GPU total: {gpu_elapsed:.1f}s, CPU total: {cpu_elapsed:.1f}s")
except Exception as exc:
    print(
        f"GPU timing comparison skipped ({type(exc).__name__}: {exc}). "
        "This is expected outside a Colab GPU runtime with qiskit-aer-gpu "
        "installed -- not an error in the main CPU sweep above."
    )